# 02.3 CNN Classification / CNN 图像分类实战

这一节把前面学的图像 shape、transform、CNN 结构真正串成一个可运行的分类项目。  
This notebook connects image shapes, transforms, and CNN structure into a runnable classification project.

这里使用 `sklearn digits` 数据集。  
We use the `sklearn digits` dataset here.

虽然它很小，但足够用来建立完整流程：  
Although it is small, it is enough to build the full workflow:

- 读入图像数据 / load image data
- 做预处理 / preprocess it
- 构建 CNN / build a CNN
- 训练和验证 / train and validate
- 在测试集上评估 / evaluate on the test set
- 对新图像做预测 / predict on new images

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 用 CNN 处理图像分类任务 / Use a CNN for image classification.
2. 把图像数据组织成 `(N, C, H, W)` / Organize image data into `(N, C, H, W)`.
3. 写出一个完整的 CNN 训练流程 / Write a complete CNN training pipeline.
4. 跟踪 train / val loss 和 accuracy / Track train / val loss and accuracy.
5. 在测试集上做最终评估 / Run a final evaluation on the test set.
6. 对单个或一小批图像做推理 / Run inference on one or a few images.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## 1. 读入数据 / Load the Data

`digits.images.shape == (N, 8, 8)`，说明它是灰度图数据。  
`digits.images.shape == (N, 8, 8)` means this is a grayscale image dataset.

In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

print("images.shape =", images.shape)
print("labels.shape =", labels.shape)
print("num classes =", len(digits.target_names))

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i], cmap="gray")
    plt.title(f"label / 标签: {labels[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. 划分训练、验证、测试 / Split Train, Validation, and Test

这里还是采用两次切分：  
We again use a two-step split:

1. 先切出测试集 / first split off the test set
2. 再从训练集中切出验证集 / then split validation from the training portion

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train:", X_train.shape)
print("val:", X_val.shape)
print("test:", X_test.shape)

## 3. 定义 transform / Define Transforms

像素范围原本大致是 `0~16`，这里先缩放到 `0~1`，再按训练集统计量做归一化。  
The original pixel range is roughly `0~16`, so we first scale it to `0~1`, then normalize it using training-set statistics.

In [ ]:
train_mean = float(X_train.mean() / 16.0)
train_std = float(X_train.std() / 16.0)

print("train_mean =", train_mean)
print("train_std =", train_std)

In [ ]:
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[train_mean], std=[train_std]),
])

## 4. 自定义图像数据集 / Custom Image Dataset

这个数据集负责两件事：  
This dataset is responsible for two things:

1. 把单张 `(H, W)` 图像变成 `(1, H, W)` / turn one `(H, W)` image into `(1, H, W)`
2. 应用 transform / apply the transform

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_ds = DigitsDataset(X_train, y_train, transform=transform)
val_ds = DigitsDataset(X_val, y_val, transform=transform)
test_ds = DigitsDataset(X_test, y_test, transform=transform)

img0, label0 = train_ds[0]
print("img0.shape =", img0.shape)
print("label0 =", label0)
print("img0.min(), img0.max() =", img0.min().item(), img0.max().item())

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## 5. 定义 CNN 模型 / Define the CNN Model

由于输入图片是 `8x8`，模型不需要很大。  
Because the input images are only `8x8`, the model does not need to be large.

In [ ]:
class DigitsCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = DigitsCNN()
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(loss_fn)
print(optimizer)

## 6. 定义训练与评估函数 / Define Training and Evaluation Functions

这里和前面的训练循环 notebook 思路一致，只是模型换成了 CNN。  
This follows the same idea as the earlier training-loop notebook, except now the model is a CNN.

In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 7. 开始训练 / Start Training

由于 `digits` 很小，这里训练 12 个 epoch 就能看到明显结果。  
Because `digits` is small, 12 epochs are enough to see clear results.

In [ ]:
history = []

for epoch in range(1, 13):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 4 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
history_df = pd.DataFrame(history)
print(history_df)

## 8. 在测试集上评估 / Evaluate on the Test Set

和之前一样，测试集只用于最终评估。  
As before, the test set is used only for the final evaluation.

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## 9. 对几张测试图像做推理 / Run Inference on a Few Test Images

这里直接拿测试集前几张图看看预测结果。  
Here we simply inspect predictions for a few test images.

In [ ]:
sample_images = []
sample_labels = []

for i in range(6):
    img, label = test_ds[i]
    sample_images.append(img)
    sample_labels.append(label.item())

sample_batch = torch.stack(sample_images)

with torch.no_grad():
    logits = model(sample_batch)
    preds = logits.argmax(dim=1).tolist()

plt.figure(figsize=(9, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].squeeze(0), cmap="gray")
    plt.title(f"true={sample_labels[i]}, pred={preds[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 练习 1 / Exercise 1
# 把 batch_size 从 32 改成 64，再重新训练一次，比较训练速度和结果。
# Change batch_size from 32 to 64, retrain, and compare training speed and results.

In [ ]:
# 练习 2 / Exercise 2
# 把第一层卷积的 out_channels 从 8 改成 16，再观察验证集表现。
# Change the first convolution's out_channels from 8 to 16, then observe validation performance.

In [ ]:
# 练习 3 / Exercise 3
# 用一句话说明为什么图像分类里输入 shape 常常是 (N, C, H, W)。
# In one sentence, explain why image classification inputs are often shaped as (N, C, H, W).

参考回答 / Reference answer:

因为一个 batch 里要同时表示样本数、通道数、高和宽，而 `PyTorch` 的卷积层默认就按 `(N, C, H, W)` 读取输入。  
Because a batch must simultaneously represent number of samples, channels, height, and width, and `PyTorch` convolution layers read inputs by default as `(N, C, H, W)`.

## 10. 小结 / Summary

到这里，你已经完成了第一个完整的 CNN 图像分类项目。  
At this point, you have completed your first full CNN image-classification project.

你已经串起来的能力 / Skills you have now connected:

- 图像 shape / image shapes
- 图像 transform / image transforms
- 自定义图像数据集 / custom image dataset
- CNN 构建 / CNN construction
- CNN 训练循环 / CNN training loop
- 测试集评估 / test-set evaluation
- 图像推理 / image inference

这就是后面迁移学习 / transfer learning 和更大图像任务的基础。  
This is the foundation for later transfer learning and larger vision tasks.